# Stage 05-D: Misinformation-Aware MIL Fusion

**Motivation:** Stage 05a used only `image_aligned_clip [64]` from COOLANT, discarding its most
valuable signal: `fake_prob` — the binary misinformation probability output of COOLANT's
`detection_module`. This notebook fixes that.

## Architecture

```
ViFactCheck article  (Statement + Evidence)
         │
         ▼
 PhoBERT-base-v2 (frozen, Stage 3.9 checkpoint)
 [CLS] token → [B, 768]    ← article-level text representation
         │
         ▼
  TextProjector: Linear[768→256] + LayerNorm + ReLU
         │
         ├─────────────────────────────────────────────────┐
         │                                                 │
         │   Per image-caption pair (n pairs per article)  │
         │     [image_aligned_clip[64], fake_prob[1]]       │
         │     = [65-dim] MIL instance                     │
         │     ▼                                           │
         │   ImageProjector: Linear[65→128] + LayerNorm    │
         │   + ReLU → [n, 128]                             │
         │     ▼                                           │
         │   MIL Attention Pooling (masked by pair count)  │
         │   → img_agg [128]   + fake_prob_agg scalar       │
         │                                                 │
         └──────── concat ──────────────────────────────────┘
                      │
              [B, 256+128=384]
                      │
         AsymmetricGate: sigmoid(Linear[384→256])
         fused = gate * text + (1-gate) * img_agg   → [B, 256]
                      │
         cat(fused, fake_prob_agg.unsqueeze(-1))    → [B, 257]
                      │
         Dropout(0.3) + Linear[257→3]
                      │
         (Real=0, Fake=1, NEI=2)
```

**Key changes vs 05a:**
- MIL instance = `[img_clip[64], fake_prob[1]]` → 65-dim (COOLANT mismatch signal per pair)
- ImageProjector: `Linear[65→128]` (smaller to fit fake_prob signal)
- Gate: `Linear[256+128=384→256]`
- Final classifier gets `fake_prob_agg` concatenated: captures article-level mismatch intensity
- Stage 2 H5 re-extracted to include `fake_probs [N]` alongside existing fields

**Baseline (Stage 05a):** val macro-F1 = 0.7947, test macro-F1 = see 05a results

**Goal:** exceed 0.80 val macro-F1

**Inputs:**
- `training/stage39_features/stage39_{train,dev,test}.h5` — PhoBERT CLS [N_articles, 768]
- `training/stage2d_features/stage2d_{train,dev,test}.h5` — COOLANT image_clip [N_pairs,64] + fake_prob [N_pairs] + article_ids

**Outputs:**
- `training/stage2d_features/stage2d_{split}.h5` — enriched Stage 2 cache with fake_prob
- `training/stage05d_cache/stage05d_{split}.h5` — MIL cache with 65-dim instances
- `training/checkpoints_stage05d/{timestamp}_mil-misinf_phobert768-coolant65_3cls_bs{B}_lr{lr}/best_model.pth`
- `training/stage05d_results/results.json`, `confusion_matrix.png`, `curves.png`


In [1]:
# ─── Environment Setup (do not edit) ─────────────────────────────────────────
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab  # noqa: F401
        return "colab", False
    except ImportError:
        pass
    if Path("/workspace").exists() and os.environ.get("VAST_CONTAINERLABEL"):
        return "vastai", False
    if Path("/workspace").exists():
        return "vastai", True
    if sys.platform == "win32":
        return "windows", False
    if sys.platform == "darwin":
        return "mac", False
    return None, True


PLATFORM, _uncertain = _detect_platform()

if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

if PLATFORM == "colab":
    PROJECT_ROOT = Path("/content/drive/MyDrive/Thesis_Final/fake-news-detection")
else:
    PROJECT_ROOT = _nb_path.parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    "colab":  PROJECT_ROOT / ".env.colab",
    "vastai": PROJECT_ROOT / ".env.vastai",
    "windows": PROJECT_ROOT / ".env.windows",
    "mac":    PROJECT_ROOT / ".env.mac",
}

if PLATFORM is None:
    _env_file = PROJECT_ROOT / ".env"
elif _uncertain:
    _env_file = _env_map["vastai"]
else:
    _env_file = _env_map[PLATFORM]

from dotenv import load_dotenv

if not _env_file.exists():
    _fallback = PROJECT_ROOT / ".env"
    if _fallback.exists():
        _env_file = _fallback
    else:
        raise FileNotFoundError(f"No .env file found. Expected: {_env_file}")
load_dotenv(_env_file, override=True)

from src.utils.env_utils import get_data_root

DATA_ROOT = get_data_root()

print(f"\u2705 Platform : {PLATFORM or 'unknown'}")
print(f"\u2705 Env file : {_env_file}")
print(f"\u2705 DATA_ROOT: {DATA_ROOT}")
print(f"{'\u2705' if DATA_ROOT.exists() else '\u274c'} Path exists: {DATA_ROOT.exists()}")

✅ Platform : mac
✅ Env file : /Users/haila/Desktop/personal-app/CascadeProjects/fake-new-detection/.env.mac
✅ DATA_ROOT: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis
✅ Path exists: True


## Step 1: Configuration

In [2]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "pipeline" else Path.cwd()
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except ImportError:
    pass
DATA_ROOT = (
    Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT
)

CONFIG = {
    "paths": {
        "project_root": PROJECT_ROOT,
        # ── Inputs from previous stages ───────────────────────────────────
        "stage39_features_dir": DATA_ROOT / "training" / "stage39_features",
        "stage39_checkpoint_root": DATA_ROOT / "training" / "checkpoints_vifactcheck",
        "stage39_manifest": None,  # None = auto-detect newest
        # COOLANT raw pairs HDF5 (from Phase 2 preprocessing)
        "coolant_hdf5_dir": DATA_ROOT / "processed_data" / "hdf5",
        # COOLANT Stage 1 checkpoint (for fake_prob extraction)
        "stage1_checkpoint_root": DATA_ROOT / "training" / "checkpoints_coolant",
        "stage1_manifest": None,  # None = auto-detect newest
        # ── Stage 2D: enriched cache (image_clip + fake_prob) ──────────────
        "stage2d_features_dir": DATA_ROOT / "training" / "stage2d_features",
        # ── Outputs ────────────────────────────────────────────────────────
        "mil_cache_dir": DATA_ROOT / "training" / "stage05d_cache",
        "checkpoint_root": DATA_ROOT / "training" / "checkpoints_stage05d",
        "results_dir": DATA_ROOT / "training" / "stage05d_results",
        "mlflow_dir": DATA_ROOT / "mlruns",
    },
    "model": {
        "arch_tag": "mil-misinf_phobert768-coolant65_3cls",
        "text_dim": 768,        # PhoBERT CLS
        "image_dim": 65,        # COOLANT image_aligned_clip[64] + fake_prob[1]
        "text_hidden": 256,     # text projection dim
        "image_hidden": 128,    # image projection dim (smaller; fake_prob is 1 scalar)
        "num_classes": 3,
        "dropout": 0.3,
        "mil_attn_type": "dot",
        "max_pairs": 8,
    },
    "training": {
        "batch_size": 32,
        "max_epochs": 40,
        "patience": 8,
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "label_smoothing": 0.1,
        "grad_clip": 1.0,
        "onecycle_pct_start": 0.05,
        "seed": 42,
    },
    "mlflow": {
        "experiment_name": "stage05d-misinformation-aware-mil",
    },
    "safety": {
        "smoke_test": False,
        "smoke_batches": 3,
        "smoke_epochs": 2,
        "auto_install_deps": False,
        "force_rebuild_stage2d": False,  # set True to re-extract fake_prob from COOLANT
        "force_rebuild_mil_cache": False,
    },
}

for key in ["stage2d_features_dir", "mil_cache_dir", "checkpoint_root", "results_dir", "mlflow_dir"]:
    CONFIG["paths"][key].mkdir(parents=True, exist_ok=True)

print(f"Project root:         {PROJECT_ROOT}")
print(f"Data root:            {DATA_ROOT}")
print(f"Stage39 features:     {CONFIG['paths']['stage39_features_dir']}")
print(f"Stage2D features:     {CONFIG['paths']['stage2d_features_dir']}")
print(f"MIL cache dir:        {CONFIG['paths']['mil_cache_dir']}")
print(f"Checkpoint root:      {CONFIG['paths']['checkpoint_root']}")
print(f"image_dim (MIL inst): {CONFIG['model']['image_dim']}  (clip[64] + fake_prob[1])")

Project root:         /Users/haila/Desktop/personal-app/CascadeProjects/fake-new-detection
Data root:            /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis
Stage39 features:     /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/stage39_features
Stage2D features:     /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/stage2d_features
MIL cache dir:        /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/stage05d_cache
Checkpoint root:      /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/checkpoints_stage05d
image_dim (MIL inst): 65  (clip[64] + fake_prob[1])


## Step 2: Dependency Preflight, Device, Seed

In [3]:
import importlib, sys, random, json, gc
from datetime import datetime
from collections import defaultdict

_required = [
    ("torch", "torch"), ("h5py", "h5py"), ("numpy", "numpy"),
    ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("seaborn", "seaborn"),
    ("tqdm", "tqdm"), ("sklearn", "scikit-learn"), ("mlflow", "mlflow"),
]
_missing = [pkg for mod, pkg in _required if importlib.util.find_spec(mod) is None]
if _missing:
    if CONFIG["safety"]["auto_install_deps"]:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install"] + _missing)
    else:
        raise RuntimeError(f"Missing packages: {_missing}")
else:
    print("All dependencies present.")

All dependencies present.


In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


def select_device():
    from src.utils.device import get_device
    return get_device()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    print(f"Seed: {seed}")


device = select_device()
seed_everything(CONFIG["training"]["seed"])
print(f"Device: {device}")

Seed: 42
Device: mps


## Step 3: Load Frozen COOLANT Stage 1 Checkpoint

Same as Stage 04 — auto-detects newest `checkpoint_manifest.json` under `checkpoints_coolant/`.
COOLANT is used purely as feature extractor to produce `fake_prob` per pair.

In [5]:
from src.models.resnet_coolant import ResNetCOOLANT
from src.preprocessing.coolant.pair_dataset import CoolantPairDataset


def resolve_stage1_checkpoint(config):
    manifest_path = config["paths"]["stage1_manifest"]
    if manifest_path is not None and Path(manifest_path).exists():
        with open(manifest_path) as f:
            m = json.load(f)
        return Path(m["best_checkpoint_path"])
    search_root = config["paths"]["stage1_checkpoint_root"]
    candidates = list(search_root.rglob("checkpoint_manifest.json"))
    if not candidates:
        raise FileNotFoundError(
            f"No checkpoint_manifest.json under {search_root}. "
            "Run 03_coolant_training.ipynb first."
        )
    newest = max(candidates, key=lambda p: p.stat().st_mtime)
    with open(newest) as f:
        m = json.load(f)
    return Path(m["best_checkpoint_path"])


def load_frozen_coolant(checkpoint_path, device):
    ckpt = torch.load(checkpoint_path, map_location=device)
    assert ckpt.get("freeze_for_stage2") is True, (
        "Phase 3 checkpoint missing freeze_for_stage2 flag — re-train Stage 1"
    )
    model = ResNetCOOLANT.from_config(ckpt["config"]["model"], device=str(device))
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model, ckpt


stage1_checkpoint_path = resolve_stage1_checkpoint(CONFIG)
stage1_model, stage1_ckpt = load_frozen_coolant(stage1_checkpoint_path, device)

total_params = sum(p.numel() for p in stage1_model.parameters())
trainable_params = sum(p.numel() for p in stage1_model.parameters() if p.requires_grad)
assert trainable_params == 0, "COOLANT must be fully frozen"

print(f"Stage 1 checkpoint: {stage1_checkpoint_path}")
print(f"Stage 1 epoch: {stage1_ckpt.get('epoch')} | val_acc: {stage1_ckpt.get('metrics', {}).get('val_accuracy')}")
print(f"Parameters — total: {total_params:,} | trainable: {trainable_params} (frozen OK)")

Stage 1 checkpoint: /Users/haila/Library/CloudStorage/GoogleDrive-latruonghai@gmail.com/My Drive/Thesis_Final/fake-news-data-for-thesis/training/checkpoints_coolant/coolant_stage1_20260610_020928/best_model.pth
Stage 1 epoch: 18 | val_acc: 0.8695
Parameters — total: 5,610,288 | trainable: 0 (frozen OK)


In [6]:
# Probe COOLANT output contract — confirm fake_prob is available
_probe_hdf5 = CONFIG["paths"]["coolant_hdf5_dir"] / "coolant_train.h5"
_probe_ds = CoolantPairDataset(str(_probe_hdf5))
_probe_loader = DataLoader(_probe_ds, batch_size=4, shuffle=False, num_workers=0)
_caption, _image, _art_ids = next(iter(_probe_loader))
_caption, _image = _caption.to(device), _image.to(device)

with torch.no_grad():
    _out = stage1_model(_caption, _image)

assert isinstance(_out, dict), f"Expected dict output, got {type(_out)}"
required_keys = ["text_aligned_clip", "image_aligned_clip", "fake_prob", "detection_logits", "attention_weights"]
missing_keys = [k for k in required_keys if k not in _out]
if missing_keys:
    raise RuntimeError(
        f"COOLANT output missing keys: {missing_keys}\n"
        "Ensure the Stage 1 model uses the full ResNetCOOLANT with detection_module."
    )

print("COOLANT output contract verified:")
for k, v in _out.items():
    print(f"  {k}: {tuple(v.shape)}")

print(f"\nfake_prob range: [{_out['fake_prob'].min().item():.4f}, {_out['fake_prob'].max().item():.4f}]")
print(f"fake_prob mean:  {_out['fake_prob'].mean().item():.4f}")

CoolantPairDataset: 6724 pairs from coolant_train.h5


RuntimeError: COOLANT output missing keys: ['fake_prob']
Ensure the Stage 1 model uses the full ResNetCOOLANT with detection_module.

## Step 4: Extract Stage 2D Features (image_clip + fake_prob)

Re-runs COOLANT inference to save `fake_probs [N]` alongside `image_features [N,64]`.
Saves to `stage2d_{split}.h5` — separate from existing `stage2_{split}.h5` to avoid breaking Stage 05a.

In [ ]:
def _apply_label_strategy(raw_label, strategy="three_class"):
    if raw_label is None:
        return None
    raw_label = int(raw_label)
    if strategy == "three_class":
        return raw_label if raw_label in (0, 1, 2) else None
    return None


def load_labels_from_hf(split):
    from datasets import load_dataset
    hf_ds = load_dataset("tranthaihoa/vifactcheck", split=split)
    articles = []
    skipped = 0
    for row in hf_ds:
        raw = row.get("labels", row.get("label", None))
        lbl = _apply_label_strategy(raw)
        articles.append({"label": lbl})
        if lbl is None:
            skipped += 1
    label_counts = {}
    for a in articles:
        if a["label"] is not None:
            label_counts[a["label"]] = label_counts.get(a["label"], 0) + 1
    print(f"[HF] {split}: {len(articles)} articles | skipped {skipped} | dist {dict(sorted(label_counts.items()))}")
    return articles


def extract_stage2d_features(split, coolant_hdf5_path, model, articles_labeled, config, device, output_path):
    """Extract image_aligned_clip + fake_prob per pair. Save to stage2d_{split}.h5."""
    if Path(output_path).exists() and not config["safety"]["force_rebuild_stage2d"]:
        print(f"[stage2d cache] {split}: using existing {output_path}")
        return

    print(f"Extracting Stage2D features for {split}...")
    ds = CoolantPairDataset(str(coolant_hdf5_path))
    loader = DataLoader(ds, batch_size=config["training"]["batch_size"], shuffle=False, num_workers=0)

    all_art_ids = ds.article_ids
    if all_art_ids.dtype.kind in ("S", "U"):
        all_art_ids = np.array([int(aid.decode() if isinstance(aid, bytes) else aid) for aid in all_art_ids])

    max_art_id = int(np.max(all_art_ids))
    if max_art_id >= len(articles_labeled):
        raise ValueError(
            f"article_id out of range: max={max_art_id} but label list has {len(articles_labeled)} entries"
        )

    image_feats, fake_probs_list, labels_list, aids_list = [], [], [], []
    skipped_null = 0

    model.eval()
    with torch.no_grad():
        for batch_idx, (caption, image, art_ids_batch) in enumerate(tqdm(loader, desc=f"  {split}")):
            if config["safety"]["smoke_test"] and batch_idx >= config["safety"]["smoke_batches"]:
                break
            caption, image = caption.to(device), image.to(device)
            out = model(caption, image)

            img_clip  = out["image_aligned_clip"].cpu().numpy().astype(np.float32)  # [B, 64]
            fake_prob = out["fake_prob"].cpu().numpy().astype(np.float32)            # [B]

            for j, art_id in enumerate(art_ids_batch.numpy()):
                art_id = int(art_id.decode() if isinstance(art_id, bytes) else art_id)
                label = articles_labeled[art_id]["label"]
                if label is None:
                    skipped_null += 1
                    continue
                image_feats.append(img_clip[j])
                fake_probs_list.append(float(fake_prob[j]))
                labels_list.append(label)
                aids_list.append(art_id)

    if skipped_null > 0:
        print(f"  \u26a0  Skipped {skipped_null} pairs with null labels")

    img_arr   = np.stack(image_feats, axis=0)              # [N, 64]
    fp_arr    = np.array(fake_probs_list, dtype=np.float32)  # [N]
    lbl_arr   = np.array(labels_list, dtype=np.int64)       # [N]
    aids_arr  = np.array(aids_list, dtype=np.int64)         # [N]

    hist = np.bincount(lbl_arr, minlength=3)
    print(f"  Pairs: {len(lbl_arr)}  |  label dist {hist.tolist()}")
    print(f"  fake_prob — min:{fp_arr.min():.4f}  mean:{fp_arr.mean():.4f}  max:{fp_arr.max():.4f}")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(output_path, "w") as f:
        f.create_dataset("image_features", data=img_arr)
        f.create_dataset("fake_probs",     data=fp_arr)
        f.create_dataset("labels",         data=lbl_arr)
        f.create_dataset("article_ids",    data=aids_arr)
        f.attrs["n_samples"]    = len(lbl_arr)
        f.attrs["num_classes"]  = 3
        f.attrs["image_dim"]    = 64
        f.attrs["has_fake_prob"] = True
        f.attrs["stage1_checkpoint"] = str(stage1_checkpoint_path)
    print(f"  Saved \u2192 {output_path}")

In [ ]:
# Run extraction for all splits
for split in ["train", "dev", "test"]:
    articles = load_labels_from_hf(split)
    extract_stage2d_features(
        split=split,
        coolant_hdf5_path=CONFIG["paths"]["coolant_hdf5_dir"] / f"coolant_{split}.h5",
        model=stage1_model,
        articles_labeled=articles,
        config=CONFIG,
        device=device,
        output_path=CONFIG["paths"]["stage2d_features_dir"] / f"stage2d_{split}.h5",
    )

print("\nStage2D extraction complete. Schema:")
for split in ["train", "dev", "test"]:
    p = CONFIG["paths"]["stage2d_features_dir"] / f"stage2d_{split}.h5"
    with h5py.File(p, "r") as f:
        print(f"  [{split}] n={f.attrs['n_samples']}")
        for k in f.keys():
            print(f"    {k}: {f[k].shape}")

## Step 5: Resolve Upstream Manifests and Validate Caches

In [ ]:
def resolve_stage39_manifest(config):
    explicit = config["paths"]["stage39_manifest"]
    if explicit is not None and Path(explicit).exists():
        with open(explicit) as f:
            return json.load(f), Path(explicit)
    search_root = config["paths"]["stage39_checkpoint_root"]
    if not search_root.exists():
        return None, None
    candidates = list(search_root.rglob("checkpoint_manifest.json"))
    if not candidates:
        return None, None
    newest = max(candidates, key=lambda p: p.stat().st_mtime)
    with open(newest) as f:
        return json.load(f), newest


stage39_manifest, stage39_manifest_path = resolve_stage39_manifest(CONFIG)
if stage39_manifest is None:
    raise RuntimeError(
        "Stage 3.9 manifest not found. Run 03.9_vifactcheck_training.ipynb first."
    )
print(f"\u2705 Stage 3.9 manifest : {stage39_manifest_path}")
print(f"   backbone           : {stage39_manifest.get('backbone')}")
print(f"   best_epoch         : {stage39_manifest.get('best_epoch')}")

s4_info = stage39_manifest.get("stage4_integration", {})
stage39_feat_dir = Path(
    s4_info.get("stage39_features_dir", str(CONFIG["paths"]["stage39_features_dir"]))
)
print(f"\n-- Stage 3.9 feature files --")
for split in ["train", "dev", "test"]:
    p = stage39_feat_dir / f"stage39_{split}.h5"
    if not p.exists():
        raise FileNotFoundError(f"Missing stage39 features: {p}")
    with h5py.File(p, "r") as f:
        print(f"  [{split}] text_features: {f['text_features'].shape}  labels: {f['labels'].shape}")

print(f"\n-- Stage2D feature files --")
for split in ["train", "dev", "test"]:
    p = CONFIG["paths"]["stage2d_features_dir"] / f"stage2d_{split}.h5"
    if not p.exists():
        raise FileNotFoundError(f"Missing stage2d features: {p} — run Step 4 first.")
    with h5py.File(p, "r") as f:
        print(f"  [{split}] image_features: {f['image_features'].shape}  fake_probs: {f['fake_probs'].shape}")

## Step 6: Build Article-Level MIL Cache (65-dim instances)

Groups Stage 2D pairs by `article_id`. Each MIL instance = `[image_clip[64], fake_prob[1]]` = 65-dim.
Attention learning will weight pairs where COOLANT detected higher mismatch.

In [ ]:
def build_mil_cache_d(
    split, stage39_h5_path, stage2d_h5_path, output_path, max_pairs, force_rebuild=False
):
    """Build MIL cache with 65-dim instances: [image_clip[64], fake_prob[1]] per pair."""
    if Path(output_path).exists() and not force_rebuild:
        print(f"[mil cache] {split}: using existing {output_path}")
        return

    print(f"Building MIL cache (stage05d) for {split}...")

    with h5py.File(stage39_h5_path, "r") as f:
        s39_text   = f["text_features"][:].astype(np.float32)  # [N_art, 768]
        s39_aids   = f["article_ids"][:].astype(np.int64)
        s39_labels = f["labels"][:].astype(np.int64)

    with h5py.File(stage2d_h5_path, "r") as f:
        s2d_image  = f["image_features"][:].astype(np.float32)  # [N_pairs, 64]
        s2d_fp     = f["fake_probs"][:].astype(np.float32)      # [N_pairs]
        s2d_aids   = f["article_ids"][:].astype(np.int64)

    # Group by article_id: (img_clip[64], fake_prob[1]) → [65]
    aid_to_pairs = defaultdict(list)  # aid -> list of [65]
    for i, aid in enumerate(s2d_aids):
        instance = np.concatenate([s2d_image[i], [s2d_fp[i]]], axis=0)  # [65]
        aid_to_pairs[int(aid)].append(instance)

    inst_dim = 65  # image_clip[64] + fake_prob[1]

    text_out, image_out, fp_agg_out, pair_counts_out, labels_out, aids_out = [], [], [], [], [], []
    skipped_no_image = 0

    for i, art_id in enumerate(s39_aids):
        art_id = int(art_id)
        pairs = aid_to_pairs.get(art_id, [])
        if len(pairs) == 0:
            skipped_no_image += 1
            continue

        n = min(len(pairs), max_pairs)
        pairs_trunc = pairs[:n]

        padded = np.zeros((max_pairs, inst_dim), dtype=np.float32)
        padded[:n] = np.stack(pairs_trunc, axis=0)

        # Aggregate fake_prob for the real pairs (mean over actual pairs)
        fp_vals = np.array([p[64] for p in pairs_trunc], dtype=np.float32)
        fp_agg  = float(fp_vals.mean())

        text_out.append(s39_text[i])
        image_out.append(padded)    # [max_pairs, 65]
        fp_agg_out.append(fp_agg)  # scalar: mean fake_prob for article
        pair_counts_out.append(n)
        labels_out.append(s39_labels[i])
        aids_out.append(art_id)

    if skipped_no_image:
        print(f"  \u26a0  Skipped {skipped_no_image} articles with no Stage2D pairs")

    text_arr   = np.stack(text_out, axis=0)              # [N, 768]
    image_arr  = np.stack(image_out, axis=0)             # [N, max_pairs, 65]
    fp_agg_arr = np.array(fp_agg_out, dtype=np.float32)  # [N]
    cnt_arr    = np.array(pair_counts_out, dtype=np.int32)
    lbl_arr    = np.array(labels_out, dtype=np.int64)
    aid_arr    = np.array(aids_out, dtype=np.int64)

    hist = np.bincount(lbl_arr, minlength=3)
    print(f"  Articles: {len(lbl_arr)}  |  label dist {hist.tolist()}")
    print(f"  Pair counts — min:{cnt_arr.min()}  mean:{cnt_arr.mean():.1f}  max:{cnt_arr.max()}")
    print(f"  fp_agg — min:{fp_agg_arr.min():.4f}  mean:{fp_agg_arr.mean():.4f}  max:{fp_agg_arr.max():.4f}")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with h5py.File(output_path, "w") as f:
        f.create_dataset("text_features",    data=text_arr)
        f.create_dataset("image_features",   data=image_arr)   # [N, max_pairs, 65]
        f.create_dataset("fake_prob_agg",    data=fp_agg_arr)  # [N] article-level mean
        f.create_dataset("pair_counts",      data=cnt_arr)
        f.create_dataset("labels",           data=lbl_arr)
        f.create_dataset("article_ids",      data=aid_arr)
        f.attrs["n_articles"] = len(lbl_arr)
        f.attrs["max_pairs"]  = max_pairs
        f.attrs["text_dim"]   = 768
        f.attrs["image_dim"]  = inst_dim   # 65
        f.attrs["num_classes"] = 3
        f.attrs["label_hist"] = str(hist.tolist())
    print(f"  Saved \u2192 {output_path}")


for split in ["train", "dev", "test"]:
    build_mil_cache_d(
        split=split,
        stage39_h5_path=stage39_feat_dir / f"stage39_{split}.h5",
        stage2d_h5_path=CONFIG["paths"]["stage2d_features_dir"] / f"stage2d_{split}.h5",
        output_path=CONFIG["paths"]["mil_cache_dir"] / f"stage05d_{split}.h5",
        max_pairs=CONFIG["model"]["max_pairs"],
        force_rebuild=CONFIG["safety"]["force_rebuild_mil_cache"],
    )
print("\n\u2705 MIL cache (stage05d) ready.")

## Step 7: Dataset and DataLoaders

In [ ]:
class MILMisinformDataset(Dataset):
    """Loads Stage 05D MIL cache: text[768], images[max_pairs,65], fake_prob_agg, label."""

    def __init__(self, h5_path):
        with h5py.File(h5_path, "r") as f:
            self.text_features = f["text_features"][:].astype(np.float32)   # [N, 768]
            self.image_features = f["image_features"][:].astype(np.float32)  # [N, max_pairs, 65]
            self.fake_prob_agg = f["fake_prob_agg"][:].astype(np.float32)   # [N]
            self.pair_counts = f["pair_counts"][:].astype(np.int32)
            self.labels = f["labels"][:].astype(np.int64)
            self.max_pairs = int(f.attrs["max_pairs"])
        print(f"MILMisinformDataset [{h5_path.name if hasattr(h5_path, 'name') else h5_path}]: "
              f"{len(self.labels)} articles, max_pairs={self.max_pairs}, inst_dim={self.image_features.shape[-1]}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.text_features[idx]),    # [768]
            torch.from_numpy(self.image_features[idx]),   # [max_pairs, 65]
            float(self.fake_prob_agg[idx]),               # scalar
            int(self.pair_counts[idx]),
            int(self.labels[idx]),
        )


def _collate(batch):
    texts, images, fp_aggs, counts, labels = zip(*batch)
    return (
        torch.stack(texts),
        torch.stack(images),
        torch.tensor(fp_aggs, dtype=torch.float32),
        torch.tensor(counts),
        torch.tensor(labels),
    )


def create_dataloaders(config, cache_dir):
    bs = config["training"]["batch_size"]
    smoke = config["safety"]["smoke_test"]

    datasets = {s: MILMisinformDataset(cache_dir / f"stage05d_{s}.h5") for s in ["train", "dev", "test"]}
    loaders = {
        "train": DataLoader(datasets["train"], batch_size=bs, shuffle=True,  num_workers=0, collate_fn=_collate),
        "dev":   DataLoader(datasets["dev"],   batch_size=bs, shuffle=False, num_workers=0, collate_fn=_collate),
        "test":  DataLoader(datasets["test"],  batch_size=bs, shuffle=False, num_workers=0, collate_fn=_collate),
    }

    if smoke:
        from itertools import islice
        class _Smoke:
            def __init__(self, l, n): self._l, self._n = l, n
            def __iter__(self): return islice(iter(self._l), self._n)
            def __len__(self): return min(self._n, len(self._l))
        return {k: _Smoke(v, config["safety"]["smoke_batches"]) for k, v in loaders.items()}, datasets

    return loaders, datasets


loaders, datasets = create_dataloaders(CONFIG, CONFIG["paths"]["mil_cache_dir"])

_b = next(iter(DataLoader(datasets["train"], batch_size=4, shuffle=False, collate_fn=_collate)))
_t, _img, _fp, _cnt, _lbl = _b
print(f"\nBatch shapes:")
print(f"  text:           {_t.shape}")
print(f"  image (65-dim): {_img.shape}")
print(f"  fake_prob_agg:  {_fp.shape}  sample: {_fp.numpy().round(4)}")
print(f"  pair_counts:    {_cnt}")
print(f"  labels:         {_lbl}")
print(f"\nTrain label hist: {np.bincount(datasets['train'].labels, minlength=3)}")

## Step 8: Misinformation-Aware MIL Fusion Head

Architecture:
- `TextProjector`: Linear[768→256] + LayerNorm + ReLU
- `ImageProjector`: Linear[65→128] + LayerNorm + ReLU (applied per pair)
- `MILAttentionPooling`: Luong tanh-dot, masked by real pair count
- `AsymmetricGate`: sigmoid(Linear[256+128=384→256])
- `Classifier`: Dropout + Linear[256+1=257→3]  ← fake_prob_agg injected directly

Why inject `fake_prob_agg` into the classifier directly (not just into MIL):
- It's an article-level signal (mean mismatch across all pairs)
- The classifier sees the "how suspicious is this article" scalar regardless of gate weighting
- This gives the model two chances to use COOLANT's mismatch signal: per-pair (via MIL attention) and aggregate (via direct injection)

In [ ]:
class MILAttentionPooling(nn.Module):
    """Luong tanh-dot attention pooling over n MIL instances."""

    def __init__(self, input_dim):
        super().__init__()
        self.W = nn.Linear(input_dim, input_dim, bias=False)
        self.v = nn.Linear(input_dim, 1, bias=False)

    def forward(self, imgs, pair_counts):
        """
        imgs:        [B, max_pairs, D]
        pair_counts: [B]
        Returns:
            aggregated: [B, D]
            attn_weights: [B, max_pairs]
        """
        B, P, D = imgs.shape
        scores = self.v(torch.tanh(self.W(imgs))).squeeze(-1)   # [B, P]
        idx  = torch.arange(P, device=imgs.device).unsqueeze(0)
        mask = idx >= pair_counts.unsqueeze(1)
        scores = scores.masked_fill(mask, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        attn = attn.masked_fill(torch.isnan(attn), 0.0)
        aggregated = (attn.unsqueeze(-1) * imgs).sum(dim=1)     # [B, D]
        return aggregated, attn


class MisinformAwareMILFusion(nn.Module):
    """
    Inputs:
        text_feat     [B, 768]             — frozen PhoBERT CLS (Stage 3.9)
        image_feat    [B, max_pairs, 65]   — COOLANT [img_clip[64], fake_prob[1]] per pair
        fake_prob_agg [B]                  — article-level mean fake_prob
        pair_counts   [B]
    Output:
        logits [B, 3]
    """

    def __init__(
        self, text_dim=768, image_dim=65, text_hidden=256, image_hidden=128,
        num_classes=3, dropout=0.3,
    ):
        super().__init__()
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, text_hidden),
            nn.LayerNorm(text_hidden),
            nn.ReLU(),
        )
        self.image_proj = nn.Sequential(
            nn.Linear(image_dim, image_hidden),
            nn.LayerNorm(image_hidden),
            nn.ReLU(),
        )
        self.mil_attn = MILAttentionPooling(image_hidden)
        # Asymmetric gate (text is primary modality)
        self.gate = nn.Linear(text_hidden + image_hidden, text_hidden)
        # Classifier gets fused[256] + fake_prob_agg[1] = 257
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(text_hidden + 1, num_classes),
        )

    def forward(self, text_feat, image_feat, fake_prob_agg, pair_counts):
        h_t = self.text_proj(text_feat)                          # [B, 256]

        B, P, D = image_feat.shape
        h_img_flat = self.image_proj(image_feat.reshape(B * P, D))  # [B*P, 128]
        h_img = h_img_flat.reshape(B, P, -1)                        # [B, P, 128]

        h_img_agg, attn_w = self.mil_attn(h_img, pair_counts)       # [B, 128]

        gate_input = torch.cat([h_t, h_img_agg], dim=-1)            # [B, 384]
        g = torch.sigmoid(self.gate(gate_input))                    # [B, 256]
        fused = g * h_t + (1.0 - g) * h_img_agg                     # [B, 256]

        # Append article-level fake_prob scalar
        fp = fake_prob_agg.unsqueeze(-1)                             # [B, 1]
        fused_with_fp = torch.cat([fused, fp], dim=-1)              # [B, 257]

        return self.classifier(fused_with_fp), attn_w


head = MisinformAwareMILFusion(
    text_dim=CONFIG["model"]["text_dim"],
    image_dim=CONFIG["model"]["image_dim"],
    text_hidden=CONFIG["model"]["text_hidden"],
    image_hidden=CONFIG["model"]["image_hidden"],
    num_classes=CONFIG["model"]["num_classes"],
    dropout=CONFIG["model"]["dropout"],
).to(device)

total_params = sum(p.numel() for p in head.parameters())
trainable_params = sum(p.numel() for p in head.parameters() if p.requires_grad)
print(f"MisinformAwareMILFusion — total: {total_params:,}  trainable: {trainable_params:,}")

head.eval()
with torch.no_grad():
    _logits, _attn = head(
        _t[:2].to(device), _img[:2].to(device),
        _fp[:2].to(device), _cnt[:2].to(device),
    )
print(f"Sanity forward — logits:{_logits.shape}  attn_weights:{_attn.shape}")
print(f"  attn sample (article 0): {_attn[0].cpu().numpy().round(4)}")

## Step 9: Loss, Optimizer, Scheduler

In [ ]:
train_labels = datasets["train"].labels
nc = CONFIG["model"]["num_classes"]
counts = np.bincount(train_labels, minlength=nc).astype(np.float64)
class_weights = torch.tensor(len(train_labels) / (nc * counts), dtype=torch.float32).to(device)
print(f"Class counts : {counts.astype(int).tolist()}")
print(f"Class weights: {class_weights.cpu().tolist()}")

loss_fn = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=CONFIG["training"]["label_smoothing"],
)

optimizer = torch.optim.AdamW(
    head.parameters(),
    lr=CONFIG["training"]["lr"],
    weight_decay=CONFIG["training"]["weight_decay"],
)

total_steps = CONFIG["training"]["max_epochs"] * len(loaders["train"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=CONFIG["training"]["lr"],
    total_steps=total_steps,
    pct_start=CONFIG["training"]["onecycle_pct_start"],
    anneal_strategy="cos",
)
print(f"Optimizer: AdamW  lr={CONFIG['training']['lr']}")
print(f"Scheduler: OneCycleLR  total_steps={total_steps}")

## Step 10: Checkpoint Naming and Save Helpers

In [ ]:
def _lr_to_str(lr):
    return f"{lr:.0e}".replace("+0", "").replace("-0", "-").replace("e0", "e")


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
arch_tag = CONFIG["model"]["arch_tag"]
bs = CONFIG["training"]["batch_size"]
lr_str = _lr_to_str(CONFIG["training"]["lr"])
run_name = f"{timestamp}_{arch_tag}_bs{bs}_lr{lr_str}"
if CONFIG["safety"]["smoke_test"]:
    run_name = f"SMOKE_{run_name}"

run_dir = CONFIG["paths"]["checkpoint_root"] / run_name
run_dir.mkdir(parents=True, exist_ok=True)
best_ckpt_path = run_dir / "best_model.pth"

print(f"Run dir: {run_dir}")


def _cfg_jsonable(cfg):
    if isinstance(cfg, dict):   return {k: _cfg_jsonable(v) for k, v in cfg.items()}
    if isinstance(cfg, Path):   return str(cfg)
    if isinstance(cfg, (list, tuple)): return [_cfg_jsonable(v) for v in cfg]
    return cfg


def save_checkpoint(path, model, epoch, metrics, config, history, is_best):
    torch.save({
        "model_state_dict": model.state_dict(),
        "config": _cfg_jsonable(config),
        "epoch": epoch, "metrics": metrics, "training_history": history,
        "arch_tag": config["model"]["arch_tag"],
        "text_dim": config["model"]["text_dim"],
        "image_dim": config["model"]["image_dim"],
        "text_hidden": config["model"]["text_hidden"],
        "image_hidden": config["model"]["image_hidden"],
        "num_classes": config["model"]["num_classes"],
        "max_pairs": config["model"]["max_pairs"],
        "is_best": is_best, "run_name": run_name,
    }, path)


def write_manifest(run_dir, best_ckpt_path, best_epoch, best_metrics, config):
    manifest = {
        "notebook": "05d_misinformation_aware_mil.ipynb",
        "proposal": "D — Misinformation-Aware MIL Fusion",
        "run_name": run_name,
        "best_checkpoint_path": str(best_ckpt_path),
        "best_epoch": best_epoch,
        "selection_metric": "val_macro_f1",
        "best_metrics": best_metrics,
        "arch": {
            "tag": config["model"]["arch_tag"],
            "text_encoder": "PhoBERT-base-v2 (frozen, Stage 3.9)",
            "text_dim": config["model"]["text_dim"],
            "image_encoder": "COOLANT [image_aligned_clip[64], fake_prob[1]] per pair",
            "image_dim": config["model"]["image_dim"],
            "image_hidden": config["model"]["image_hidden"],
            "fusion": "MIL attention pooling (65-dim) + asymmetric gate + fake_prob_agg injection",
            "max_pairs": config["model"]["max_pairs"],
            "num_classes": config["model"]["num_classes"],
        },
        "training": _cfg_jsonable(config["training"]),
        "baseline_05a": {"val_macro_f1": 0.7947, "best_epoch": 10},
    }
    p = run_dir / "checkpoint_manifest.json"
    with open(p, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"Manifest written: {p}")


print("Checkpoint helpers defined.")

## Step 11: Training Loop

In [ ]:
def compute_metrics(y_true, y_pred, prefix, nc):
    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    per_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    m = {f"{prefix}_accuracy": round(acc, 4), f"{prefix}_macro_f1": round(macro_f1, 4)}
    for i, name in enumerate(["real", "fake", "nei"]):
        m[f"{prefix}_f1_{name}"] = round(float(per_f1[i]) if i < len(per_f1) else 0.0, 4)
    return m


def run_epoch(model, loader, loss_fn, optimizer, scheduler, device, config, train):
    model.train() if train else model.eval()
    total_loss, n_batches = 0.0, 0
    y_true_all, y_pred_all = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        pbar = tqdm(loader, desc="  train" if train else "  eval ", leave=False)
        for batch in pbar:
            text_feat, image_feat, fake_prob_agg, pair_counts, labels = batch
            text_feat     = text_feat.to(device)
            image_feat    = image_feat.to(device)
            fake_prob_agg = fake_prob_agg.to(device)
            pair_counts   = pair_counts.to(device)
            labels        = labels.to(device)

            logits, _ = model(text_feat, image_feat, fake_prob_agg, pair_counts)
            loss = loss_fn(logits, labels)

            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss: {loss.item()}")

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), config["training"]["grad_clip"])
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            total_loss += loss.item()
            n_batches  += 1
            preds = logits.argmax(dim=-1).cpu().numpy()
            y_pred_all.extend(preds.tolist())
            y_true_all.extend(labels.cpu().numpy().tolist())
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(1, n_batches), np.array(y_true_all), np.array(y_pred_all)


print("Training functions defined.")

In [ ]:
import mlflow

mlflow_enabled = False
try:
    mlflow.set_tracking_uri(CONFIG["paths"]["mlflow_dir"].as_uri())
    mlflow.set_experiment(CONFIG["mlflow"]["experiment_name"])
    _run = mlflow.start_run(run_name=run_name)
    mlflow.log_params({
        "arch_tag": CONFIG["model"]["arch_tag"],
        "text_dim":    CONFIG["model"]["text_dim"],
        "image_dim":   CONFIG["model"]["image_dim"],
        "text_hidden": CONFIG["model"]["text_hidden"],
        "image_hidden": CONFIG["model"]["image_hidden"],
        "max_pairs": CONFIG["model"]["max_pairs"],
        "batch_size":  CONFIG["training"]["batch_size"],
        "max_epochs":  CONFIG["training"]["max_epochs"],
        "lr":          CONFIG["training"]["lr"],
        "seed":        CONFIG["training"]["seed"],
    })
    mlflow_enabled = True
    print(f"MLflow run: {_run.info.run_id}")
except Exception as e:
    print(f"MLflow disabled ({e})")

In [ ]:
best_val_f1 = -1.0
best_epoch  = -1
best_metrics = {}
patience_ctr = 0
history = []
max_epochs = (
    CONFIG["safety"]["smoke_epochs"]
    if CONFIG["safety"]["smoke_test"]
    else CONFIG["training"]["max_epochs"]
)

print(f"Training for up to {max_epochs} epochs (patience={CONFIG['training']['patience']})")
print(f"Checkpoint dir: {run_dir}")
print(f"Baseline (05a): val macro-F1 = 0.7947")

for epoch in range(1, max_epochs + 1):
    train_loss, yt, yp = run_epoch(
        head, loaders["train"], loss_fn, optimizer, scheduler, device, CONFIG, train=True
    )
    val_loss, yv_t, yv_p = run_epoch(
        head, loaders["dev"], loss_fn, None, None, device, CONFIG, train=False
    )

    train_m = compute_metrics(yt, yp, "train", CONFIG["model"]["num_classes"])
    val_m   = compute_metrics(yv_t, yv_p, "val",   CONFIG["model"]["num_classes"])
    row = {"epoch": epoch, "train_loss": round(train_loss, 4), "val_loss": round(val_loss, 4), **train_m, **val_m}
    history.append(row)

    if mlflow_enabled:
        mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss, **train_m, **val_m}, step=epoch)

    val_f1 = val_m["val_macro_f1"]
    marker = ""
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch
        best_metrics = val_m
        patience_ctr = 0
        save_checkpoint(best_ckpt_path, head, epoch, val_m, CONFIG, history, is_best=True)
        delta = val_f1 - 0.7947
        marker = f" \u2190 best  (\u0394 vs 05a: {delta:+.4f})"
    else:
        patience_ctr += 1

    print(
        f"Epoch {epoch:02d}/{max_epochs} | "
        f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f} | "
        f"val_macro_f1={val_f1:.4f}  val_acc={val_m['val_accuracy']:.4f}"
        f"{marker}"
    )

    if patience_ctr >= CONFIG["training"]["patience"]:
        print(f"Early stopping at epoch {epoch} (patience={CONFIG['training']['patience']})")
        break

write_manifest(run_dir, best_ckpt_path, best_epoch, best_metrics, CONFIG)
if mlflow_enabled:
    mlflow.log_metrics({f"best_{k}": v for k, v in best_metrics.items()})
    mlflow.end_run()

delta_vs_05a = best_val_f1 - 0.7947
print(f"\nBest epoch: {best_epoch}  |  val macro-F1: {best_val_f1:.4f}  (\u0394 vs 05a: {delta_vs_05a:+.4f})")
print(f"Best val metrics: {best_metrics}")

## Step 12: Test Evaluation and Results Export

In [ ]:
ckpt = torch.load(best_ckpt_path, map_location=device)
head.load_state_dict(ckpt["model_state_dict"])
head.eval()

test_loss, yt_t, yt_p = run_epoch(
    head, loaders["test"], loss_fn, None, None, device, CONFIG, train=False
)
test_m = compute_metrics(yt_t, yt_p, "test", CONFIG["model"]["num_classes"])

print(f"\nTest results (best checkpoint from epoch {best_epoch}):")
for k, v in test_m.items():
    print(f"  {k}: {v}")

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(yt_t, yt_p, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", ax=ax,
    xticklabels=["Real", "Fake", "NEI"],
    yticklabels=["Real", "Fake", "NEI"],
    cmap="Blues",
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(
    f"Stage 05-D: Misinformation-Aware MIL\n"
    f"Test macro-F1={test_m['test_macro_f1']:.4f}  (05a: 0.7947 val)"
)
fig.tight_layout()
cm_path = CONFIG["paths"]["results_dir"] / f"{run_name}_confusion_matrix.png"
fig.savefig(cm_path, dpi=150)
plt.show()
print(f"Saved: {cm_path}")

# ── Export results JSON ────────────────────────────────────────────────────────
results = {
    "notebook": "05d_misinformation_aware_mil.ipynb",
    "proposal": "D — Misinformation-Aware MIL Fusion",
    "run_name": run_name,
    "best_epoch": best_epoch,
    "val_metrics": best_metrics,
    "test_metrics": test_m,
    "arch": {
        "tag": CONFIG["model"]["arch_tag"],
        "text_encoder": "PhoBERT-base-v2 (frozen, Stage 3.9)",
        "image_encoder": "COOLANT [image_aligned_clip[64], fake_prob[1]] per pair (Stage 2D)",
        "fusion": "MIL attention pooling (65-dim) + asymmetric gate [384→256] + fake_prob_agg injection",
        "max_pairs": CONFIG["model"]["max_pairs"],
        "text_hidden": CONFIG["model"]["text_hidden"],
        "image_hidden": CONFIG["model"]["image_hidden"],
    },
    "baseline_05a": {"val_macro_f1": 0.7947, "best_epoch": 10},
    "delta_val_vs_05a": round(best_val_f1 - 0.7947, 4),
    "training_history": history,
}
results_path = CONFIG["paths"]["results_dir"] / f"{run_name}_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved: {results_path}")

## Step 13: Training Curves

In [ ]:
df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df["epoch"], df["train_loss"], label="train")
axes[0].plot(df["epoch"], df["val_loss"],   label="val")
axes[0].axvline(best_epoch, color="red", linestyle="--", label=f"best (ep {best_epoch})")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(df["epoch"], df["train_macro_f1"], label="train")
axes[1].plot(df["epoch"], df["val_macro_f1"],   label="val")
axes[1].axvline(best_epoch, color="red", linestyle="--")
axes[1].axhline(0.7947, color="gray", linestyle=":", label="05a baseline")
axes[1].set_title("Macro F1")
axes[1].legend()

fig.suptitle(f"Stage 05-D: Misinformation-Aware MIL — {run_name}")
fig.tight_layout()
curve_path = CONFIG["paths"]["results_dir"] / f"{run_name}_curves.png"
fig.savefig(curve_path, dpi=150)
plt.show()
print(f"Saved: {curve_path}")

## Step 14: Attention Weight Analysis

Visualizes how attention weights correlate with `fake_prob` values.
If the architecture is working, pairs with higher `fake_prob` should get higher attention weight.

In [ ]:
# Collect attention weights and fake_prob values from dev set
head.eval()
all_attn, all_fp, all_counts, all_labels = [], [], [], []

with torch.no_grad():
    for batch in DataLoader(datasets["dev"], batch_size=64, shuffle=False, collate_fn=_collate):
        text_f, img_f, fp_agg, cnt, lbl = batch
        text_f = text_f.to(device)
        img_f  = img_f.to(device)
        fp_agg = fp_agg.to(device)
        cnt    = cnt.to(device)

        _, attn_w = head(text_f, img_f, fp_agg, cnt)
        all_attn.append(attn_w.cpu().numpy())
        all_fp.append(img_f[:, :, 64].cpu().numpy())  # fake_prob channel per pair
        all_counts.append(cnt.cpu().numpy())
        all_labels.append(lbl.numpy())

all_attn   = np.concatenate(all_attn,   axis=0)  # [N, max_pairs]
all_fp     = np.concatenate(all_fp,     axis=0)  # [N, max_pairs]
all_counts = np.concatenate(all_counts, axis=0)  # [N]
all_labels = np.concatenate(all_labels, axis=0)  # [N]

# For articles with multiple pairs: compute correlation between attn and fake_prob
correlations = []
for i in range(len(all_counts)):
    n = int(all_counts[i])
    if n < 2:
        continue
    fp_vals  = all_fp[i, :n]
    attn_vals = all_attn[i, :n]
    if fp_vals.std() > 1e-6 and attn_vals.std() > 1e-6:
        corr = np.corrcoef(fp_vals, attn_vals)[0, 1]
        correlations.append(corr)

if correlations:
    mean_corr = np.mean(correlations)
    print(f"Attention vs fake_prob correlation (dev, n_pairs>=2):")
    print(f"  n_articles: {len(correlations)}")
    print(f"  mean_corr:  {mean_corr:.4f}  (positive = attention follows mismatch)")
    print(f"  median_corr: {np.median(correlations):.4f}")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(correlations, bins=30, color="#4F46E5", alpha=0.8)
    ax.axvline(mean_corr, color="red", linestyle="--", label=f"mean={mean_corr:.3f}")
    ax.axvline(0, color="gray", linestyle=":")
    ax.set_xlabel("Pearson corr(attn_weight, fake_prob) per article")
    ax.set_ylabel("Count")
    ax.set_title("Do high-attention pairs have higher fake_prob?\n(positive = model learns mismatch weighting)")
    ax.legend()
    fig.tight_layout()
    corr_path = CONFIG["paths"]["results_dir"] / f"{run_name}_attn_fp_correlation.png"
    fig.savefig(corr_path, dpi=150)
    plt.show()
    print(f"Saved: {corr_path}")
else:
    print("Not enough multi-pair articles for correlation analysis.")

## Step 15: Summary

In [ ]:
print("=" * 68)
print("Stage 05-D: Misinformation-Aware MIL Fusion — Complete")
print("=" * 68)
print(f"""\nArchitecture:
  Text:  PhoBERT CLS [768] → Linear[256] + LayerNorm
  Image: [img_clip[64], fake_prob[1]] per pair → Linear[128] + MIL-Attn
  Gate:  sigmoid(Linear[384→256]) asymmetric text-primary
  Head:  Dropout + Linear[257→3]  (fake_prob_agg injected)\n""")
print(f"Best epoch:      {best_epoch}")
print(f"Val macro-F1:    {best_val_f1:.4f}  (05a baseline: 0.7947  \u0394={best_val_f1-0.7947:+.4f})")
print(f"Test macro-F1:   {test_m['test_macro_f1']:.4f}")
print(f"Test accuracy:   {test_m['test_accuracy']:.4f}")
print(f"Test F1-real:    {test_m['test_f1_real']:.4f}")
print(f"Test F1-fake:    {test_m['test_f1_fake']:.4f}")
print(f"Test F1-nei:     {test_m['test_f1_nei']:.4f}")
print()
print(f"Checkpoint: {best_ckpt_path}")
print(f"Results:    {results_path}")

if best_val_f1 > 0.7947:
    print(f"\n\u2705 Exceeds 05a baseline! Delta = +{best_val_f1-0.7947:.4f}")
else:
    print(f"\n\u26a0  Did not exceed 05a baseline. Consider tuning lr, image_hidden, or dropout.")